In [6]:
from sklearn.datasets import fetch_openml

## 위험 고객 분류

In [2]:
import numpy as np
import pandas as pd
import joblib

# KKBox 점수·이탈 모델
kk = pd.read_csv("../../data/processed/kkbox_scored.csv")
saved = joblib.load("../../models/lgbm_final.joblib")

churn_model = saved["model"]
TH = saved["threshold"]
CHURN_FEATS = saved["features"]
CHURN_CATS = saved["categories"]

for c in CHURN_CATS:
    kk[c] = kk[c].astype("category")

# 팀원 군집 모델·군집용 피처
cb = joblib.load("../../models/cluster_model.joblib")
cf = pd.read_csv("../../data/processed/cluster_features_all.csv")

# 검증용: 팀원 군집 결과
clusters = pd.read_csv("../../data/processed/risk_clusters.csv")

kk.shape, cf.shape, clusters.shape, TH, len(cb["features"])

((100000, 37), (100000, 19), (6696, 3), 0.2823849351481365, 18)

In [3]:
risk_users = kk[kk["risk"] == 1].copy()

Xc = risk_users[["msno"]].merge(cf, on="msno", how="left")
Xc.index = risk_users.index

feat = cb["features"]
ok = Xc[feat].notna().all(axis=1)

Xt = Xc.loc[ok, feat].copy()
for c in cb["log_features"]:
    Xt[c] = np.log1p(Xt[c])

Xs = pd.DataFrame(cb["scaler"].transform(Xt), columns=feat, index=Xt.index)
Xs = Xs.drop(columns=cb["drop_after_scale"])

risk_users["segment"] = "미분류"
risk_users.loc[Xs.index, "segment"] = pd.Series(cb["kmeans"].predict(Xs), index=Xs.index).map(cb["names"])

risk_users["segment"].value_counts()

segment
저활동·단기 구독형        3450
장기 플랜·고결제 고위험형    1747
장기 관계·고빈도 거래형      822
미분류                820
반복 거래·취소 위험형       678
Name: count, dtype: int64

In [4]:
check = risk_users[["msno", "segment"]].merge(clusters[["msno", "cluster_name"]], on="msno")
len(check), (check["segment"] == check["cluster_name"]).mean()

(6327, np.float64(1.0))

In [5]:
SEGMENT_NAMES = [
    "저활동·단기 구독형",
    "반복 거래·취소 위험형",
    "장기 플랜·고결제 고위험형",
    "장기 관계·고빈도 거래형",
]

summary = risk_users[risk_users["segment"].isin(SEGMENT_NAMES)].groupby("segment").agg(
    인원=("msno", "size"),
    평균_이탈확률=("churn_prob", "mean"),
    실제_이탈률=("is_churn", "mean"),
    평균_활동일=("activity_days", "mean"),
    마지막접속후_평균일수=("days_since_last_log", "mean"),
    평균_해지횟수=("cancel_count", "mean"),
    평균_결제금액=("avg_payment", "mean"),
).reindex(SEGMENT_NAMES).round(3)
summary

,인원,평균_이탈확률,실제_이탈률,평균_활동일,마지막접속후_평균일수,평균_해지횟수,평균_결제금액
segment,,,,,,,
저활동·단기 구독형,3450,0.449,0.459,27.118,2.901,0.013,198.452
반복 거래·취소 위험형,678,0.740,0.751,42.254,12.770,1.383,125.267
장기 플랜·고결제 고위험형,1747,0.844,0.863,60.396,1.887,0.004,1021.222
장기 관계·고빈도 거래형,822,0.572,0.573,53.203,1.642,0.938,182.884


In [6]:
risk_segments = risk_users[["msno", "segment", "churn_prob"]].reset_index(drop=True)

risk_segment_summary = summary.drop(columns=["실제_이탈률"]).reset_index()

risk_segments.to_csv("../../data/processed/risk_segments.csv", index=False)
risk_segment_summary.to_csv("../../data/processed/risk_segment_summary.csv", index=False)

risk_segments.shape, risk_segments["segment"].value_counts()



((7517, 3),
 segment
 저활동·단기 구독형        3450
 장기 플랜·고결제 고위험형    1747
 장기 관계·고빈도 거래형      822
 미분류                820
 반복 거래·취소 위험형       678
 Name: count, dtype: int64)

In [123]:
risk_users["renew"] = risk_users["last_auto_renew"].map({0: "자동갱신 끔", 1: "자동갱신 켬"}).fillna("거래 기록 없음")
pd.crosstab(risk_users["renew"], risk_users["risk_type"], margins=True)

risk_type,결제·해지형,활동량 저하형,휴면형,All
renew,,,,
거래 기록 없음,1,132,5,138
자동갱신 끔,2308,2575,1307,6190
자동갱신 켬,945,3,241,1189
All,3254,2710,1553,7517


In [1]:
len(set(clusters["msno"]) & set(risk_users["msno"])), len(clusters)

NameError: name 'clusters' is not defined

In [ ]:
SEGMENT_NAMES = [
    "저활동·단기 구독형",
    "반복 거래·취소 위험형",
    "장기 플랜·고결제 고위험형",
    "장기 관계·고빈도 거래형",
]

summary = risk_users[risk_users["segment"].isin(SEGMENT_NAMES)].groupby("segment").agg(
    인원=("msno", "size"),
    평균_이탈확률=("churn_prob", "mean"),
    실제_이탈률=("is_churn", "mean"),
    평균_활동일=("activity_days", "mean"),
    마지막접속후_평균일수=("days_since_last_log", "mean"),
    평균_해지횟수=("cancel_count", "mean"),
    평균_결제금액=("avg_payment", "mean"),
).reindex(SEGMENT_NAMES).round(3)
summary

,인원,평균_이탈확률,실제_이탈률,평균_활동일,마지막접속후_평균일수,로그_있음_비율,평균_해지횟수,평균_결제금액
segment,,,,,,,,
자동갱신 끔 · 결제·해지형,2308,0.743,0.740,65.370,0.318,1.000,0.134,752.842
자동갱신 끔 · 활동량 저하형,2575,0.478,0.502,18.002,1.132,0.923,0.203,174.280
자동갱신 끔 · 휴면형,1307,0.552,0.562,32.950,8.825,0.986,0.161,368.127
자동갱신 켬 · 결제·해지형,945,0.765,0.797,30.861,6.337,0.644,1.322,115.975
자동갱신 켬 · 휴면형,241,0.754,0.763,8.207,41.562,0.465,1.033,81.569


In [127]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 화면에서 고객별 분류를 연결할 때 사용하는 파일
risk_segments = (
    risk_users.loc[risk_users["segment"] != "제외", ["msno", "segment", "churn_prob"]]
    .reset_index(drop=True)
)

# 화면의 분류 카드에 표시할 요약 파일 (실제 이탈 여부는 운영 화면에서 제외)
risk_segment_summary = (
    summary.drop(columns=["실제_이탈률"], errors="ignore")
    .reset_index()
)

segments_path = OUTPUT_DIR / "risk_segments.csv"
summary_path = OUTPUT_DIR / "risk_segment_summary.csv"

risk_segments.to_csv(segments_path, index=False, encoding="utf-8-sig")
risk_segment_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print(f"고객 분류 저장: {segments_path} ({len(risk_segments):,}명)")
print(f"분류 요약 저장: {summary_path} ({len(risk_segment_summary):,}개 분류)")

고객 분류 저장: ..\..\data\processed\risk_segments.csv (7,376명)
분류 요약 저장: ..\..\data\processed\risk_segment_summary.csv (5개 분류)


In [128]:
print(risk_segments.shape)
display(risk_segments.head())
display(risk_segments["segment"].value_counts())

print(risk_segment_summary.shape)
display(risk_segment_summary)

(7376, 3)


,msno,segment,churn_prob
0,UghFAbnjl2mzoe3BOwQkvKRg6sd/qf55wi2Iz4VPr7w=,자동갱신 켬 · 결제·해지형,0.960156
1,qJhTE7thVJXybPLtb+5ys6SdifMmFqTOy9xvo0XGgME=,자동갱신 끔 · 활동량 저하형,0.961439
2,xkc7YJYS41r+PVmVU+HHeBWxBM8Cdf6PUXqrYASbOdY=,자동갱신 켬 · 휴면형,0.960337
3,2Z9BKbhhTUdORZY/J51tHeYMjZwfHG0el3RwlF9NN+I=,자동갱신 켬 · 결제·해지형,0.954458
4,2u4wnVMNFgfPzofa+V5yP0+D8SVWUR7rrRhBqhc4BIM=,자동갱신 끔 · 휴면형,0.835936


segment
자동갱신 끔 · 활동량 저하형    2575
자동갱신 끔 · 결제·해지형     2308
자동갱신 끔 · 휴면형        1307
자동갱신 켬 · 결제·해지형      945
자동갱신 켬 · 휴면형         241
Name: count, dtype: int64

(5, 8)


,segment,인원,평균_이탈확률,평균_활동일,마지막접속후_평균일수,로그_있음_비율,평균_해지횟수,평균_결제금액
0,자동갱신 끔 · 결제·해지형,2308,0.743,65.370,0.318,1.000,0.134,752.842
1,자동갱신 끔 · 활동량 저하형,2575,0.478,18.002,1.132,0.923,0.203,174.280
2,자동갱신 끔 · 휴면형,1307,0.552,32.950,8.825,0.986,0.161,368.127
3,자동갱신 켬 · 결제·해지형,945,0.765,30.861,6.337,0.644,1.322,115.975
4,자동갱신 켬 · 휴면형,241,0.754,8.207,41.562,0.465,1.033,81.569


## starbucks 데이터에 따른 고객 행동 변화

In [129]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
)
from sklearn.model_selection import GroupKFold, cross_val_predict


STARBUCKS_DIR = Path("../../data/raw/starbucks")
MODEL_DIR = Path("../../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PORTFOLIO_PATH = STARBUCKS_DIR / "portfolio.json"
PROFILE_PATH = STARBUCKS_DIR / "profile.json"
TRANSCRIPT_PATH = STARBUCKS_DIR / "transcript.json"

for path in [PORTFOLIO_PATH, PROFILE_PATH, TRANSCRIPT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

print("Starbucks 데이터 경로 확인 완료")

Starbucks 데이터 경로 확인 완료


In [130]:
portfolio = pd.read_json(
    PORTFOLIO_PATH,
    orient="records",
    lines=True,
)

profile = pd.read_json(
    PROFILE_PATH,
    orient="records",
    lines=True,
)

transcript = pd.read_json(
    TRANSCRIPT_PATH,
    orient="records",
    lines=True,
)

print("portfolio :", portfolio.shape)
print("profile   :", profile.shape)
print("transcript:", transcript.shape)

display(portfolio)
display(profile.head())
display(transcript.head())

portfolio : (10, 6)
profile   : (17000, 5)
transcript: (306534, 4)


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7
5,3,"[web, email, mobile, social]",7,7,discount,2298d6c36e964ae4a3e7e9706d1fb8c2
6,2,"[web, email, mobile, social]",10,10,discount,fafdcd668e3743c1bb461111dcafc2a4
7,0,"[email, mobile, social]",0,3,informational,5a8bc65990b245e5a138643cd4eb9837
8,5,"[web, email, mobile, social]",5,5,bogo,f19421c1d4aa40978ebb69ca19b0e20d
9,2,"[web, email, mobile]",10,7,discount,2906b810c7d4411798c6938adc9daaa5


,gender,age,id,became_member_on,income
0,None,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,None,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,None,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [131]:
CHANNELS = ["email", "mobile", "social", "web"]

for channel in CHANNELS:
    portfolio[channel] = portfolio["channels"].apply(
        lambda channels: int(channel in channels)
    )

# 고객이 달성해야 하는 구매 조건 대비 보상 비율
portfolio["benefit"] = np.where(
    portfolio["difficulty"] > 0,
    portfolio["reward"] / portfolio["difficulty"],
    0.0,
)

portfolio_check = portfolio[
    [
        "id",
        "offer_type",
        "reward",
        "difficulty",
        "benefit",
        "duration",
        *CHANNELS,
    ]
].copy()

display(portfolio_check)

,id,offer_type,reward,difficulty,benefit,duration,email,mobile,social,web
0,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,10,1.000000,7,1,1,1,0
1,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,10,1.000000,5,1,1,1,1
2,3f207df678b143eea3cee63160fa8bed,informational,0,0,0.000000,4,1,1,0,1
3,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,5,1.000000,7,1,1,0,1
4,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,5,20,0.250000,10,1,0,0,1
5,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,3,7,0.428571,7,1,1,1,1
6,fafdcd668e3743c1bb461111dcafc2a4,discount,2,10,0.200000,10,1,1,1,1
7,5a8bc65990b245e5a138643cd4eb9837,informational,0,0,0.000000,3,1,1,1,0
8,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5,1.000000,5,1,1,1,1
9,2906b810c7d4411798c6938adc9daaa5,discount,2,10,0.200000,7,1,1,0,1


In [132]:
    # 118세는 Starbucks 데이터에서 결측치를 나타내는 값
profile["age"] = profile["age"].replace(118, np.nan)

profile["became_member_on"] = pd.to_datetime(
    profile["became_member_on"].astype(str),
    format="%Y%m%d",
)

# 가장 최근 가입일을 기준으로 가입기간 계산
reference_date = profile["became_member_on"].max()

profile["tenure_days"] = (
    reference_date - profile["became_member_on"]
).dt.days

profile[
    ["age", "gender", "tenure_days"]
].describe(include="all")

,age,gender,tenure_days
count,14825.000000,14825,17000.000000
unique,NaN,3,NaN
top,NaN,M,NaN
freq,NaN,8484,NaN
mean,54.393524,NaN,517.449882
std,17.383705,NaN,411.223904
min,18.000000,NaN,0.000000
25%,42.000000,NaN,208.000000
50%,55.000000,NaN,358.000000
75%,66.000000,NaN,791.000000


In [133]:
transcript_value = pd.json_normalize(transcript["value"])

empty_offer_id = pd.Series(
    pd.NA,
    index=transcript_value.index,
    dtype="object",
)

offer_id_space = (
    transcript_value["offer id"]
    if "offer id" in transcript_value.columns
    else empty_offer_id
)

offer_id_under = (
    transcript_value["offer_id"]
    if "offer_id" in transcript_value.columns
    else empty_offer_id
)

transcript["offer_id"] = offer_id_space.combine_first(offer_id_under)

transcript["amount"] = (
    transcript_value["amount"]
    if "amount" in transcript_value.columns
    else np.nan
)

transcript["day"] = transcript["time"] / 24

display(transcript["event"].value_counts())
print(
    "캠페인 발송 시점:",
    sorted(
        transcript.loc[
            transcript["event"] == "offer received",
            "time",
        ].unique()
    ),
)

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

캠페인 발송 시점: [np.int64(0), np.int64(168), np.int64(336), np.int64(408), np.int64(504), np.int64(576)]


In [134]:
# 캠페인 발송 이후 행동을 관찰하는 시간
W = 72  # 72시간 = 3일

waves = sorted(
    transcript.loc[
        transcript["event"] == "offer received",
        "time",
    ].unique()
)

received = transcript.loc[
    transcript["event"] == "offer received",
    ["person", "time", "offer_id"],
].copy()

transactions = transcript.loc[
    transcript["event"] == "transaction",
    ["person", "time", "amount"],
].copy()

panel_rows = []

for wave in waves:
    # 모든 고객을 해당 발송 회차의 기본 대상으로 생성
    wave_panel = profile[["id"]].rename(
        columns={"id": "person"}
    )
    wave_panel["wave"] = wave

    # 해당 시점에 실제로 받은 캠페인
    wave_offer = (
        received.loc[
            received["time"] == wave,
            ["person", "offer_id"],
        ]
        .drop_duplicates("person")
    )

    wave_panel = wave_panel.merge(
        wave_offer,
        on="person",
        how="left",
    )

    # 발송 후 72시간 동안의 거래
    wave_transactions = (
        transactions.loc[
            (transactions["time"] >= wave)
            & (transactions["time"] < wave + W)
        ]
        .groupby("person")
        .agg(
            n_tx=("amount", "size"),
            spend=("amount", "sum"),
        )
        .reset_index()
    )

    wave_panel = wave_panel.merge(
        wave_transactions,
        on="person",
        how="left",
    )

    wave_panel[["n_tx", "spend"]] = (
        wave_panel[["n_tx", "spend"]].fillna(0)
    )

    panel_rows.append(wave_panel)

panel = pd.concat(
    panel_rows,
    ignore_index=True,
)

print("기본 패널:", panel.shape)
display(panel.head())

기본 패널: (102000, 5)


,person,wave,offer_id,n_tx,spend
0,68be06ca386d4c31939f3a4f0e3dd783,0,NaN,0.0,0.00
1,0610b486422d4921ae7d2bf64640c50b,0,NaN,1.0,21.51
2,38fe809add3b4fcf9315a9694bb96ff5,0,NaN,0.0,0.00
3,78afa995795e4d85b5d9ceeca43f5fef,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,0.0,0.00
4,a03223e636434f42ac4c3df47e8bac43,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,0.0,0.00


In [135]:
offer_columns = [
    "id",
    "offer_type",
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

panel = (
    panel.merge(
        portfolio[offer_columns],
        left_on="offer_id",
        right_on="id",
        how="left",
    )
    .drop(columns="id")
)

# 캠페인을 받지 않은 고객
panel["offer_type"] = panel["offer_type"].fillna("none")

offer_value_columns = [
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

panel[offer_value_columns] = (
    panel[offer_value_columns].fillna(0)
)

# income은 의도적으로 결합하지 않음
panel = (
    panel.merge(
        profile[
            [
                "id",
                "age",
                "gender",
                "tenure_days",
            ]
        ],
        left_on="person",
        right_on="id",
        how="left",
    )
    .drop(columns="id")
)

print("최종 패널:", panel.shape)

panel_summary = (
    panel.groupby(
        "offer_type",
        observed=True,
    )
    .agg(
        데이터수=("person", "size"),
        평균_거래횟수=("n_tx", "mean"),
        평균_거래금액=("spend", "mean"),
    )
    .round(3)
)

display(panel_summary)

최종 패널: (102000, 15)


,데이터수,평균_거래횟수,평균_거래금액
offer_type,,,
bogo,30499,1.046,13.397
discount,30543,0.962,12.612
informational,15235,0.900,11.281
none,25723,0.594,7.575


In [136]:
BEHAVIOR_CATS = [
    "gender",
    "offer_type",
]

BEHAVIOR_CUSTOMER_FEATS = [
    "age",
    "gender",
    "tenure_days",
]

BEHAVIOR_OFFER_FEATS = [
    "offer_type",
    "email",
    "mobile",
    "social",
    "web",
    "benefit",
    "duration",
]

BEHAVIOR_FEATS = (
    BEHAVIOR_CUSTOMER_FEATS
    + BEHAVIOR_OFFER_FEATS
)

for column in BEHAVIOR_CATS:
    panel[column] = panel[column].astype("category")

print("행동 모델 피처:", BEHAVIOR_FEATS)
print("범주형 피처:", BEHAVIOR_CATS)

display(
    panel[
        BEHAVIOR_FEATS + ["n_tx"]
    ].head()
)

행동 모델 피처: ['age', 'gender', 'tenure_days', 'offer_type', 'email', 'mobile', 'social', 'web', 'benefit', 'duration']
범주형 피처: ['gender', 'offer_type']


,age,gender,tenure_days,offer_type,email,mobile,social,web,benefit,duration,n_tx
0,NaN,NaN,529,none,0.0,0.0,0.0,0.0,0.00,0.0,0.0
1,55.0,F,376,none,0.0,0.0,0.0,0.0,0.00,0.0,1.0
2,NaN,NaN,14,none,0.0,0.0,0.0,0.0,0.00,0.0,0.0
3,75.0,F,443,bogo,1.0,1.0,0.0,1.0,1.00,7.0,0.0
4,NaN,NaN,356,discount,1.0,0.0,0.0,1.0,0.25,10.0,0.0


In [137]:
behavior_model = LGBMRegressor(
    objective="poisson",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    random_state=42,
    verbosity=-1,
)

group_kfold = GroupKFold(
    n_splits=5
)

oof_prediction = cross_val_predict(
    behavior_model,
    panel[BEHAVIOR_FEATS],
    panel["n_tx"],
    cv=group_kfold,
    groups=panel["person"],
)

# Poisson deviance 계산을 위해 0보다 크게 제한
oof_prediction = np.clip(
    oof_prediction,
    1e-8,
    None,
)

mae = mean_absolute_error(
    panel["n_tx"],
    oof_prediction,
)

rmse = np.sqrt(
    mean_squared_error(
        panel["n_tx"],
        oof_prediction,
    )
)

poisson_deviance = mean_poisson_deviance(
    panel["n_tx"],
    oof_prediction,
)

print(f"GroupKFold MAE             : {mae:.4f}")
print(f"GroupKFold RMSE            : {rmse:.4f}")
print(f"GroupKFold Poisson deviance: {poisson_deviance:.4f}")

GroupKFold MAE             : 0.7747
GroupKFold RMSE            : 0.9948
GroupKFold Poisson deviance: 1.1694


In [138]:
behavior_validation = (
    panel.assign(
        predicted_n_tx=oof_prediction
    )
    .groupby(
        "offer_type",
        observed=True,
    )
    .agg(
        데이터수=("person", "size"),
        실제_평균거래=("n_tx", "mean"),
        예측_평균거래=("predicted_n_tx", "mean"),
    )
)

behavior_validation["차이"] = (
    behavior_validation["예측_평균거래"]
    - behavior_validation["실제_평균거래"]
)

display(
    behavior_validation.round(3)
)

,데이터수,실제_평균거래,예측_평균거래,차이
offer_type,,,,
bogo,30499,1.046,1.045,-0.001
discount,30543,0.962,0.958,-0.003
informational,15235,0.900,0.902,0.002
none,25723,0.594,0.598,0.004


In [139]:
behavior_model.fit(
    panel[BEHAVIOR_FEATS],
    panel["n_tx"],
)

print("행동 모델 최종 학습 완료")

행동 모델 최종 학습 완료


In [140]:
behavior_categories = {
    column: panel[column].cat.categories.tolist()
    for column in BEHAVIOR_CATS
}

behavior_bundle = {
    "model": behavior_model,
    "features": BEHAVIOR_FEATS,
    "categorical_features": BEHAVIOR_CATS,
    "categories": behavior_categories,
    "target": "n_tx",
    "observation_hours": W,
}

BEHAVIOR_MODEL_PATH = (
    MODEL_DIR / "behavior_model.joblib"
)

joblib.dump(
    behavior_bundle,
    BEHAVIOR_MODEL_PATH,
)

print(
    f"행동 모델 저장 완료: {BEHAVIOR_MODEL_PATH}"
)
print("저장 피처:", behavior_bundle["features"])
print("저장 범주:", behavior_bundle["categories"])

행동 모델 저장 완료: ..\..\models\behavior_model.joblib
저장 피처: ['age', 'gender', 'tenure_days', 'offer_type', 'email', 'mobile', 'social', 'web', 'benefit', 'duration']
저장 범주: {'gender': ['F', 'M', 'O'], 'offer_type': ['bogo', 'discount', 'informational', 'none']}


In [141]:
loaded_behavior = joblib.load(
    BEHAVIOR_MODEL_PATH
)

assert (
    loaded_behavior["features"]
    == BEHAVIOR_FEATS
)

assert set(
    loaded_behavior["categories"]
) == set(BEHAVIOR_CATS)

test_prediction = loaded_behavior[
    "model"
].predict(
    panel[BEHAVIOR_FEATS].head(10)
)

print("모델 재로딩 확인 완료")
print("예측값:", test_prediction.round(3))

모델 재로딩 확인 완료
예측값: [0.597 0.598 0.444 0.873 0.534 0.422 0.741 0.7   0.486 0.595]


## 4. 분류 고객 캠페인 시뮬레이션

In [142]:
def select_segment(segment_name):
    segment_ids = risk_segments.loc[
        risk_segments["segment"] == segment_name,
        "msno",
    ]

    selected = kk.loc[
        kk["msno"].isin(segment_ids)
    ].copy()

    return selected

In [143]:
test_segment_name = "자동갱신 끔 · 활동량 저하형"

selected_kk = select_segment(
    test_segment_name
)

print("선택 집단:", test_segment_name)
print("고객 수:", len(selected_kk))

display(
    selected_kk[
        [
            "msno",
            "churn_prob",
            "bd_clean",
            "gender_status",
            "tenure_date",
            "activity_days",
            "total_secs",
            "days_since_last_log",
        ]
    ].head()
)

선택 집단: 자동갱신 끔 · 활동량 저하형
고객 수: 2575


,msno,churn_prob,bd_clean,gender_status,tenure_date,activity_days,total_secs,days_since_last_log
19,qJhTE7thVJXybPLtb+5ys6SdifMmFqTOy9xvo0XGgME=,0.961439,NaN,미입력,2403.0,0.0,0.000,NaN
118,nAvn64vTNrReeTkZ0uIFE+FHYZp9d+EYJ44zjqI2Vog=,0.562884,21.0,male,611.0,3.0,2040.301,4.0
177,dLacMPNtZ+lOyK6Wy6mJbqpOUJkZV5cxslqRbMvKVT4=,0.491130,24.0,male,799.0,2.0,30511.421,0.0
206,zYsQ7IxKxdv/mfaBVJ3Pja1p4PpjSpjHGeuAcrrYiRk=,0.296952,20.0,male,641.0,58.0,1317324.820,0.0
230,TcR0Vq3rSqLobCkutLi1lt43MRWhL+ol1rabmI5d3pY=,0.304935,16.0,male,62.0,23.0,41696.526,0.0


In [144]:
def make_behavior_customers(selected_customers):
    behavior_customers = pd.DataFrame(
        {
            "age": selected_customers["bd_clean"],
            "gender": selected_customers["gender_status"].map(
                {
                    "male": "M",
                    "female": "F",
                }
            ),
            "tenure_days": selected_customers["tenure_date"],
        },
        index=selected_customers.index,
    )

    return behavior_customers

In [145]:
test_behavior_customers = make_behavior_customers(
    selected_kk
)

print(test_behavior_customers.shape)
display(test_behavior_customers.head())
display(test_behavior_customers.isna().mean().round(3))

(2575, 3)


,age,gender,tenure_days
19,NaN,NaN,2403.0
118,21.0,M,611.0
177,24.0,M,799.0
206,20.0,M,641.0
230,16.0,M,62.0


age            0.398
gender         0.391
tenure_days    0.009
dtype: float64

In [146]:
OFFER_TYPE_MAP = {
    "구독료 할인": "discount",
    "무료 이용 혜택": "bogo",
    "콘텐츠 추천": "informational",
    "단순 안내": "informational",
}

CHANNEL_MAP = {
    "이메일": "email",
    "앱 푸시": "mobile",
    "문자": "mobile",
    "웹": "web",
    "소셜": "social",
}


def campaign_to_offer(
    channel,
    offer_name,
    benefit_pct,
    duration,
):
    if channel not in CHANNEL_MAP:
        raise ValueError(f"지원하지 않는 발송 채널: {channel}")

    if offer_name not in OFFER_TYPE_MAP:
        raise ValueError(f"지원하지 않는 오퍼 유형: {offer_name}")

    offer_type = OFFER_TYPE_MAP[offer_name]

    offer = {
        "offer_type": offer_type,
        "email": 0,
        "mobile": 0,
        "social": 0,
        "web": 0,
        "benefit": 0.0,
        "duration": int(duration),
    }

    offer[CHANNEL_MAP[channel]] = 1

    if offer_type == "discount":
        offer["benefit"] = benefit_pct / 100

    elif offer_type == "bogo":
        # Starbucks의 BOGO는 조건금액과 보상금액이 같음
        offer["benefit"] = 1.0

    return offer

In [147]:
test_offer = campaign_to_offer(
    channel="앱 푸시",
    offer_name="구독료 할인",
    benefit_pct=30,
    duration=7,
)

test_offer

{'offer_type': 'discount',
 'email': 0,
 'mobile': 1,
 'social': 0,
 'web': 0,
 'benefit': 0.3,
 'duration': 7}

In [148]:
def behavior_lift(
    behavior_customers,
    offer,
    behavior_bundle,
):
    model = behavior_bundle["model"]
    features = behavior_bundle["features"]
    categories = behavior_bundle["categories"]

    campaign_on = behavior_customers.copy()
    campaign_off = behavior_customers.copy()

    # 캠페인을 적용한 입력
    for column, value in offer.items():
        campaign_on[column] = value

    # 아무 캠페인도 적용하지 않은 입력
    campaign_off["offer_type"] = "none"

    for channel in [
        "email",
        "mobile",
        "social",
        "web",
    ]:
        campaign_off[channel] = 0

    campaign_off["benefit"] = 0.0
    campaign_off["duration"] = 0

    # 학습 당시와 동일한 범주 순서 적용
    for column, category_values in categories.items():
        campaign_on[column] = pd.Categorical(
            campaign_on[column],
            categories=category_values,
        )

        campaign_off[column] = pd.Categorical(
            campaign_off[column],
            categories=category_values,
        )

    prediction_on = np.clip(
        model.predict(campaign_on[features]),
        1e-8,
        None,
    )

    prediction_off = np.clip(
        model.predict(campaign_off[features]),
        1e-8,
        None,
    )

    result = pd.DataFrame(
        {
            "behavior_before": prediction_off,
            "behavior_after": prediction_on,
            "lift": prediction_on / prediction_off,
        },
        index=behavior_customers.index,
    )

    return result

In [149]:
test_behavior_result = behavior_lift(
    behavior_customers=test_behavior_customers,
    offer=test_offer,
    behavior_bundle=loaded_behavior,
)

display(test_behavior_result.head())

display(
    test_behavior_result.describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    ).round(3)
)

,behavior_before,behavior_after,lift
19,0.606486,0.858614,1.415721
118,1.000565,1.175173,1.174510
177,1.071284,1.320095,1.232254
206,1.024370,1.175173,1.147216
230,0.469290,0.494823,1.054409


,behavior_before,behavior_after,lift
count,2575.000,2575.000,2575.000
mean,0.715,0.891,1.234
std,0.218,0.300,0.154
min,0.365,0.374,0.812
1%,0.398,0.402,0.978
5%,0.408,0.412,1.004
25%,0.551,0.609,1.103
50%,0.606,0.859,1.256
75%,0.932,1.186,1.339
95%,1.046,1.263,1.466


In [150]:
def apply_behavior(
    churn_features,
    lift,
):
    after = churn_features.copy()

    lift = pd.Series(
        lift,
        index=after.index,
        dtype=float,
    )

    if lift.isna().any():
        raise ValueError(
            "일부 고객의 행동 변화율을 찾을 수 없습니다."
        )

    # 총 청취시간
    if "total_secs" in after.columns:
        after["total_secs"] = (
            after["total_secs"]
            .astype(float)
            .mul(lift)
            .clip(lower=0)
        )

    # 완청 곡 수와 고유 곡 수
    count_columns = [
        "total_num_100",
        "total_num_unq",
    ]

    for column in count_columns:
        if column in after.columns:
            after[column] = (
                after[column]
                .astype(float)
                .mul(lift)
                .round()
                .clip(lower=0)
            )

    # 분석 기간이 최대 90일이므로 90일로 제한
    if "activity_days" in after.columns:
        after["activity_days"] = (
            after["activity_days"]
            .astype(float)
            .mul(lift)
            .round()
            .clip(lower=0, upper=90)
        )

    # 실제 로그가 있는 고객만 마지막 접속일 변경
    if {
        "days_since_last_log",
        "has_log",
    }.issubset(after.columns):

        has_log_mask = after["has_log"].eq(1)

        after.loc[
            has_log_mask,
            "days_since_last_log",
        ] = (
            after.loc[
                has_log_mask,
                "days_since_last_log",
            ]
            .astype(float)
            .div(lift.loc[has_log_mask])
            .round()
            .clip(lower=0)
        )

    # 활동일 변화에 맞춰 일평균 피처 재계산
    derived_features = {
        "avg_daily_secs": "total_secs",
        "avg_daily_complete": "total_num_100",
        "avg_daily_unq": "total_num_unq",
    }

    if "activity_days" in after.columns:
        valid_days = after["activity_days"] > 0

        for derived, total in derived_features.items():
            if (
                derived in after.columns
                and total in after.columns
            ):
                after.loc[
                    valid_days,
                    derived,
                ] = (
                    after.loc[valid_days, total]
                    / after.loc[
                        valid_days,
                        "activity_days",
                    ]
                )

                # 원래 활동이 없는 고객은 기존 값을 유지
                after.loc[
                    ~valid_days,
                    derived,
                ] = churn_features.loc[
                    ~valid_days,
                    derived,
                ]

    return after

In [151]:
X_before = selected_kk[
    CHURN_FEATS
].copy()

X_after = apply_behavior(
    churn_features=X_before,
    lift=test_behavior_result["lift"],
)

print("변경 전:", X_before.shape)
print("변경 후:", X_after.shape)

display(
    pd.DataFrame(
        {
            "활동일_전": X_before["activity_days"].head(),
            "활동일_후": X_after["activity_days"].head(),
            "청취시간_전": X_before["total_secs"].head(),
            "청취시간_후": X_after["total_secs"].head(),
            "마지막접속_전": X_before["days_since_last_log"].head(),
            "마지막접속_후": X_after["days_since_last_log"].head(),
        }
    )
)

변경 전: (2575, 33)
변경 후: (2575, 33)


,활동일_전,활동일_후,청취시간_전,청취시간_후,마지막접속_전,마지막접속_후
19,0.0,0.0,0.000,0.000000e+00,NaN,NaN
118,3.0,4.0,2040.301,2.396353e+03,4.0,3.0
177,2.0,2.0,30511.421,3.759783e+04,0.0,0.0
206,58.0,67.0,1317324.820,1.511256e+06,0.0,0.0
230,23.0,24.0,41696.526,4.396519e+04,0.0,0.0


In [152]:
# 최종 이탈 모델이 계산한 캠페인 적용 전·후 확률
model_prob_before = churn_model.predict_proba(
    X_before
)[:, 1]

model_prob_after = churn_model.predict_proba(
    X_after
)[:, 1]

# 같은 모델 안에서 발생한 캠페인의 순수한 확률 변화량
model_probability_change = (
    model_prob_after
    - model_prob_before
)

# 위험 고객 분류에 사용한 교차검증·테스트 기준확률
prob_before = selected_kk[
    "churn_prob"
].to_numpy()

# 기존 기준확률에 캠페인 변화량만 적용
raw_prob_after = (
    prob_before
    + model_probability_change
)

# 확률은 0~1 범위로 제한
prob_after = np.clip(
    raw_prob_after,
    0,
    1,
)

clipped_count = (
    (raw_prob_after < 0)
    | (raw_prob_after > 1)
).sum()

print(
    "기준 평균 이탈확률:",
    prob_before.mean(),
)

print(
    "모델 기준 평균 변화량:",
    model_probability_change.mean(),
)

print(
    "캠페인 후 평균 이탈확률:",
    prob_after.mean(),
)

print(
    "0~1 범위로 제한된 고객 수:",
    clipped_count,
)

기준 평균 이탈확률: 0.4779049896123396
모델 기준 평균 변화량: -0.024407856422915392
캠페인 후 평균 이탈확률: 0.4534971331894243
0~1 범위로 제한된 고객 수: 0


In [153]:
simulation_detail = pd.DataFrame(
    {
        "msno": selected_kk["msno"].values,
        "segment": test_segment_name,
        "lift": test_behavior_result["lift"].values,

        # 위험 고객 분류에 사용한 기준확률
        "prob_before": prob_before,

        # 최종 모델이 계산한 전후 참고값
        "model_prob_before": model_prob_before,
        "model_prob_after": model_prob_after,

        # 캠페인으로 발생한 확률 변화량
        "model_probability_change": (
            model_probability_change
        ),

        # 기준확률에 변화량을 적용한 최종 결과
        "prob_after": prob_after,
    },
    index=selected_kk.index,
)

simulation_detail["churn_reduction"] = (
    simulation_detail["prob_before"]
    - simulation_detail["prob_after"]
)

display(
    simulation_detail[
        [
            "msno",
            "lift",
            "prob_before",
            "model_probability_change",
            "prob_after",
            "churn_reduction",
        ]
    ].head()
)

,msno,lift,prob_before,model_probability_change,prob_after,churn_reduction
19,qJhTE7thVJXybPLtb+5ys6SdifMmFqTOy9xvo0XGgME=,1.415721,0.961439,0.000000,0.961439,0.000000
118,nAvn64vTNrReeTkZ0uIFE+FHYZp9d+EYJ44zjqI2Vog=,1.174510,0.562884,-0.029756,0.533127,0.029756
177,dLacMPNtZ+lOyK6Wy6mJbqpOUJkZV5cxslqRbMvKVT4=,1.232254,0.491130,-0.019802,0.471328,0.019802
206,zYsQ7IxKxdv/mfaBVJ3Pja1p4PpjSpjHGeuAcrrYiRk=,1.147216,0.296952,-0.029543,0.267409,0.029543
230,TcR0Vq3rSqLobCkutLi1lt43MRWhL+ol1rabmI5d3pY=,1.054409,0.304935,0.000000,0.304935,0.000000


In [154]:
expected_churn_before = (
    simulation_detail["prob_before"].sum()
)

expected_churn_after = (
    simulation_detail["prob_after"].sum()
)

reduced_customers = (
    expected_churn_before
    - expected_churn_after
)

churn_change_rate = (
    expected_churn_after
    / expected_churn_before
    - 1
) * 100

simulation_summary = pd.DataFrame(
    [
        {
            "분류": test_segment_name,
            "고객수": len(simulation_detail),

            "평균_행동변화율(%)": (
                simulation_detail["lift"].mean()
                - 1
            ) * 100,

            "현재_평균이탈확률": (
                simulation_detail[
                    "prob_before"
                ].mean()
            ),

            "캠페인후_평균이탈확률": (
                simulation_detail[
                    "prob_after"
                ].mean()
            ),

            "현재_예상이탈자": (
                expected_churn_before
            ),

            "캠페인후_예상이탈자": (
                expected_churn_after
            ),

            "예상_감소인원": (
                reduced_customers
            ),

            "예상이탈_변화율(%)": (
                churn_change_rate
            ),
        }
    ]
)

display(
    simulation_summary.round(
        {
            "평균_행동변화율(%)": 2,
            "현재_평균이탈확률": 4,
            "캠페인후_평균이탈확률": 4,
            "현재_예상이탈자": 2,
            "캠페인후_예상이탈자": 2,
            "예상_감소인원": 2,
            "예상이탈_변화율(%)": 2,
        }
    )
)

,분류,고객수,평균_행동변화율(%),현재_평균이탈확률,캠페인후_평균이탈확률,현재_예상이탈자,캠페인후_예상이탈자,예상_감소인원,예상이탈_변화율(%)
0,자동갱신 끔 · 활동량 저하형,2575,23.37,0.4779,0.4535,1230.61,1167.76,62.85,-5.11


In [156]:
feature_labels = {
    "activity_days": "평균 활동일",
    "total_secs": "평균 총 청취시간",
    "total_num_100": "평균 완청 곡 수",
    "total_num_unq": "평균 고유 곡 수",
    "days_since_last_log": "마지막 접속 후 평균일수",
}

feature_changes = []

for column, label in feature_labels.items():
    if (
        column not in X_before.columns
        or column not in X_after.columns
    ):
        continue

    before_mean = X_before[column].mean()
    after_mean = X_after[column].mean()

    if before_mean != 0:
        change_pct = (
            after_mean
            / before_mean
            - 1
        ) * 100
    else:
        change_pct = np.nan

    feature_changes.append(
        {
            "행동 지표": label,
            "캠페인 전": before_mean,
            "캠페인 후": after_mean,
            "변화율(%)": change_pct,
        }
    )

feature_change_table = pd.DataFrame(
    feature_changes
)

display(
    feature_change_table.round(2)
)

,행동 지표,캠페인 전,캠페인 후,변화율(%)
0,평균 활동일,18.00,21.90,21.66
1,평균 총 청취시간,145102.38,176795.33,21.84
2,평균 완청 곡 수,554.36,675.81,21.91
3,평균 고유 곡 수,526.94,641.07,21.66
4,마지막 접속 후 평균일수,1.13,0.94,-16.54


### 4-8. 분류 × 마케팅 시뮬레이션 함수

In [158]:
def simulate_segment(
    segment_name,
    channel,
    offer_name,
    benefit_pct,
    duration,
):
    """
    선택한 위험 고객 분류에 캠페인을 적용하고
    행동 변화와 이탈확률 변화를 계산한다.

    Parameters
    ----------
    segment_name : str
        적용할 위험 고객 분류 이름
    channel : str
        이메일, 앱 푸시, 문자, 웹, 소셜
    offer_name : str
        구독료 할인, 무료 이용 혜택, 콘텐츠 추천, 단순 안내
    benefit_pct : float
        할인율. 0~50 사이 값
    duration : int
        캠페인 적용 기간. 3~10일
    """

    if not 0 <= benefit_pct <= 50:
        raise ValueError(
            "혜택률은 0~50 사이로 입력해야 합니다."
        )

    if not 3 <= duration <= 10:
        raise ValueError(
            "캠페인 기간은 3~10일 사이로 입력해야 합니다."
        )

    # 1. 선택한 위험 고객 분류 추출
    selected_customers = select_segment(
        segment_name
    )

    if selected_customers.empty:
        raise ValueError(
            f"선택한 분류에 고객이 없습니다: {segment_name}"
        )

    # 2. Starbucks 행동 모델 입력 생성
    behavior_customers = make_behavior_customers(
        selected_customers
    )

    # 3. 선택한 캠페인 정보 생성
    offer = campaign_to_offer(
        channel=channel,
        offer_name=offer_name,
        benefit_pct=benefit_pct,
        duration=duration,
    )

    # 4. 캠페인 적용 전후 행동 변화율 계산
    behavior_result = behavior_lift(
        behavior_customers=behavior_customers,
        offer=offer,
        behavior_bundle=loaded_behavior,
    )

    # 5. KKBOX 이탈 모델 입력 준비
    X_before = selected_customers[
        CHURN_FEATS
    ].copy()

    X_after = apply_behavior(
        churn_features=X_before,
        lift=behavior_result["lift"],
    )

    # 6. 동일한 이탈 모델에서 전후 확률 변화량 계산
    model_prob_before = (
        churn_model.predict_proba(
            X_before
        )[:, 1]
    )

    model_prob_after = (
        churn_model.predict_proba(
            X_after
        )[:, 1]
    )

    model_probability_change = (
        model_prob_after
        - model_prob_before
    )

    # 7. 위험 고객 분류에 사용한 기준확률
    prob_before = selected_customers[
        "churn_prob"
    ].to_numpy()

    # 기준확률에 캠페인으로 인한 변화량만 적용
    raw_prob_after = (
        prob_before
        + model_probability_change
    )

    prob_after = np.clip(
        raw_prob_after,
        0,
        1,
    )

    # 8. 고객별 상세 결과
    simulation_detail = pd.DataFrame(
        {
            "msno": selected_customers[
                "msno"
            ].values,

            "segment": segment_name,

            "lift": behavior_result[
                "lift"
            ].values,

            "prob_before": prob_before,

            "model_prob_before": (
                model_prob_before
            ),

            "model_prob_after": (
                model_prob_after
            ),

            "model_probability_change": (
                model_probability_change
            ),

            "prob_after": prob_after,
        },
        index=selected_customers.index,
    )

    simulation_detail[
        "churn_reduction"
    ] = (
        simulation_detail["prob_before"]
        - simulation_detail["prob_after"]
    )

    # 9. 분류 전체 이탈 변화 요약
    expected_churn_before = (
        simulation_detail[
            "prob_before"
        ].sum()
    )

    expected_churn_after = (
        simulation_detail[
            "prob_after"
        ].sum()
    )

    reduced_customers = (
        expected_churn_before
        - expected_churn_after
    )

    if expected_churn_before > 0:
        churn_change_rate = (
            expected_churn_after
            / expected_churn_before
            - 1
        ) * 100
    else:
        churn_change_rate = np.nan

    simulation_summary = pd.DataFrame(
        [
            {
                "분류": segment_name,
                "발송채널": channel,
                "혜택유형": offer_name,
                "혜택률(%)": benefit_pct,
                "기간(일)": duration,

                "고객수": len(
                    simulation_detail
                ),

                "평균_행동변화율(%)": (
                    simulation_detail[
                        "lift"
                    ].mean()
                    - 1
                ) * 100,

                "현재_평균이탈확률": (
                    simulation_detail[
                        "prob_before"
                    ].mean()
                ),

                "캠페인후_평균이탈확률": (
                    simulation_detail[
                        "prob_after"
                    ].mean()
                ),

                "현재_예상이탈자": (
                    expected_churn_before
                ),

                "캠페인후_예상이탈자": (
                    expected_churn_after
                ),

                "예상_감소인원": (
                    reduced_customers
                ),

                "예상이탈_변화율(%)": (
                    churn_change_rate
                ),
            }
        ]
    )

    # 10. 행동 지표 변화표
    feature_labels = {
        "activity_days": "평균 활동일",
        "total_secs": "평균 총 청취시간",
        "total_num_100": "평균 완청 곡 수",
        "total_num_unq": "평균 고유 곡 수",
        "days_since_last_log": (
            "마지막 접속 후 평균일수"
        ),
    }

    feature_changes = []

    for column, label in feature_labels.items():
        if (
            column not in X_before.columns
            or column not in X_after.columns
        ):
            continue

        before_mean = X_before[
            column
        ].mean()

        after_mean = X_after[
            column
        ].mean()

        if before_mean != 0:
            change_pct = (
                after_mean
                / before_mean
                - 1
            ) * 100
        else:
            change_pct = np.nan

        feature_changes.append(
            {
                "행동 지표": label,
                "캠페인 전": before_mean,
                "캠페인 후": after_mean,
                "변화율(%)": change_pct,
            }
        )

    feature_change_table = pd.DataFrame(
        feature_changes
    )

    return {
        "summary": simulation_summary,
        "feature_changes": feature_change_table,
        "detail": simulation_detail,
    }

In [159]:
test_result = simulate_segment(
    segment_name="자동갱신 끔 · 활동량 저하형",
    channel="앱 푸시",
    offer_name="구독료 할인",
    benefit_pct=30,
    duration=7,
)

display(
    test_result["summary"].round(
        {
            "평균_행동변화율(%)": 2,
            "현재_평균이탈확률": 4,
            "캠페인후_평균이탈확률": 4,
            "현재_예상이탈자": 2,
            "캠페인후_예상이탈자": 2,
            "예상_감소인원": 2,
            "예상이탈_변화율(%)": 2,
        }
    )
)

display(
    test_result[
        "feature_changes"
    ].round(2)
)

display(
    test_result["detail"][
        [
            "msno",
            "lift",
            "prob_before",
            "model_probability_change",
            "prob_after",
            "churn_reduction",
        ]
    ].head()
)

,분류,발송채널,혜택유형,혜택률(%),기간(일),고객수,평균_행동변화율(%),현재_평균이탈확률,캠페인후_평균이탈확률,현재_예상이탈자,캠페인후_예상이탈자,예상_감소인원,예상이탈_변화율(%)
0,자동갱신 끔 · 활동량 저하형,앱 푸시,구독료 할인,30,7,2575,23.37,0.4779,0.4535,1230.61,1167.76,62.85,-5.11


,행동 지표,캠페인 전,캠페인 후,변화율(%)
0,평균 활동일,18.00,21.90,21.66
1,평균 총 청취시간,145102.38,176795.33,21.84
2,평균 완청 곡 수,554.36,675.81,21.91
3,평균 고유 곡 수,526.94,641.07,21.66
4,마지막 접속 후 평균일수,1.13,0.94,-16.54


,msno,lift,prob_before,model_probability_change,prob_after,churn_reduction
19,qJhTE7thVJXybPLtb+5ys6SdifMmFqTOy9xvo0XGgME=,1.415721,0.961439,0.000000,0.961439,0.000000
118,nAvn64vTNrReeTkZ0uIFE+FHYZp9d+EYJ44zjqI2Vog=,1.174510,0.562884,-0.029756,0.533127,0.029756
177,dLacMPNtZ+lOyK6Wy6mJbqpOUJkZV5cxslqRbMvKVT4=,1.232254,0.491130,-0.019802,0.471328,0.019802
206,zYsQ7IxKxdv/mfaBVJ3Pja1p4PpjSpjHGeuAcrrYiRk=,1.147216,0.296952,-0.029543,0.267409,0.029543
230,TcR0Vq3rSqLobCkutLi1lt43MRWhL+ol1rabmI5d3pY=,1.054409,0.304935,0.000000,0.304935,0.000000


### 4-9. 동일 캠페인의 5개 위험 고객 분류 비교

In [160]:
# 5개 분류에 동일한 캠페인을 적용하여
# 고객 분류별 반응 차이를 비교한다.

segment_results = {}
segment_summaries = []

for segment_name in SEGMENTS.keys():
    result = simulate_segment(
        segment_name=segment_name,
        channel="앱 푸시",
        offer_name="구독료 할인",
        benefit_pct=30,
        duration=7,
    )

    # 다음 분석에서 상세 결과를 사용할 수 있도록 저장
    segment_results[
        segment_name
    ] = result

    segment_summaries.append(
        result["summary"]
    )

# 5개 분류의 요약 결과를 하나의 표로 합침
segment_comparison = pd.concat(
    segment_summaries,
    ignore_index=True,
)

comparison_columns = [
    "분류",
    "고객수",
    "평균_행동변화율(%)",
    "현재_평균이탈확률",
    "캠페인후_평균이탈확률",
    "현재_예상이탈자",
    "캠페인후_예상이탈자",
    "예상_감소인원",
    "예상이탈_변화율(%)",
]

display(
    segment_comparison[
        comparison_columns
    ].round(
        {
            "평균_행동변화율(%)": 2,
            "현재_평균이탈확률": 4,
            "캠페인후_평균이탈확률": 4,
            "현재_예상이탈자": 2,
            "캠페인후_예상이탈자": 2,
            "예상_감소인원": 2,
            "예상이탈_변화율(%)": 2,
        }
    )
)

,분류,고객수,평균_행동변화율(%),현재_평균이탈확률,캠페인후_평균이탈확률,현재_예상이탈자,캠페인후_예상이탈자,예상_감소인원,예상이탈_변화율(%)
0,자동갱신 끔 · 활동량 저하형,2575,23.37,0.4779,0.4535,1230.61,1167.76,62.85,-5.11
1,자동갱신 끔 · 결제·해지형,2308,28.35,0.7427,0.7387,1714.16,1705.00,9.16,-0.53
2,자동갱신 끔 · 휴면형,1307,24.44,0.5516,0.5329,720.96,696.49,24.46,-3.39
3,자동갱신 켬 · 결제·해지형,945,28.07,0.7648,0.7556,722.73,714.01,8.72,-1.21
4,자동갱신 켬 · 휴면형,241,20.63,0.7538,0.7511,181.67,181.03,0.65,-0.36


### 4-10. 자동갱신 켬 · 휴면형 효과 점검

In [162]:
target_segment = "자동갱신 켬 · 휴면형"

target_customers = select_segment(
    target_segment
)

target_detail = segment_results[
    target_segment
]["detail"].copy()

probability_unchanged = np.isclose(
    target_detail[
        "model_probability_change"
    ],
    0,
    atol=1e-12,
)

dormant_diagnostic = pd.DataFrame(
    [
        {
            "분류": target_segment,

            "고객수": len(
                target_customers
            ),

            "로그있음_비율(%)": (
                target_customers[
                    "has_log"
                ].eq(1).mean()
                * 100
            ),

            "활동일0_비율(%)": (
                target_customers[
                    "activity_days"
                ].fillna(0).eq(0).mean()
                * 100
            ),

            "청취시간0_비율(%)": (
                target_customers[
                    "total_secs"
                ].fillna(0).eq(0).mean()
                * 100
            ),

            "이탈확률_무변화비율(%)": (
                probability_unchanged.mean()
                * 100
            ),

            "평균_행동변화율(%)": (
                target_detail[
                    "lift"
                ].mean()
                - 1
            ) * 100,

            "예상_감소인원": (
                target_detail[
                    "churn_reduction"
                ].sum()
            ),
        }
    ]
)

display(
    dormant_diagnostic.round(2)
)

,분류,고객수,로그있음_비율(%),활동일0_비율(%),청취시간0_비율(%),이탈확률_무변화비율(%),평균_행동변화율(%),예상_감소인원
0,자동갱신 켬 · 휴면형,241,46.47,53.53,53.53,75.52,20.63,0.65


In [163]:
dormant_customer_check = pd.DataFrame(
    {
        "로그 상태": np.where(
            target_customers[
                "has_log"
            ].eq(1),
            "로그 있음",
            "로그 없음",
        ),

        "활동일": target_customers[
            "activity_days"
        ],

        "총 청취시간": target_customers[
            "total_secs"
        ],

        "마지막 접속 후 경과일": (
            target_customers[
                "days_since_last_log"
            ]
        ),

        "이탈 감소확률": target_detail[
            "churn_reduction"
        ],

        "이탈확률 변화 없음": (
            probability_unchanged
        ),
    },
    index=target_customers.index,
)

log_status_comparison = (
    dormant_customer_check
    .groupby("로그 상태")
    .agg(
        고객수=(
            "이탈 감소확률",
            "size",
        ),

        평균_활동일=(
            "활동일",
            "mean",
        ),

        평균_청취시간=(
            "총 청취시간",
            "mean",
        ),

        평균_마지막접속경과일=(
            "마지막 접속 후 경과일",
            "mean",
        ),

        예상_감소인원=(
            "이탈 감소확률",
            "sum",
        ),

        이탈확률_무변화비율=(
            "이탈확률 변화 없음",
            "mean",
        ),
    )
    .reset_index()
)

log_status_comparison[
    "이탈확률_무변화비율(%)"
] = (
    log_status_comparison[
        "이탈확률_무변화비율"
    ]
    * 100
)

log_status_comparison = (
    log_status_comparison.drop(
        columns=[
            "이탈확률_무변화비율"
        ]
    )
)

display(
    log_status_comparison.round(2)
)

,로그 상태,고객수,평균_활동일,평균_청취시간,평균_마지막접속경과일,예상_감소인원,이탈확률_무변화비율(%)
0,로그 없음,129,0.00,0.00,NaN,0.00,100.00
1,로그 있음,112,17.66,141484.38,41.56,0.65,47.32


### 4-11. 위험 고객 분류 × 마케팅 유형 비교

In [165]:
# Starbucks 행동 모델에서 서로 다르게 인식되는
# 대표 마케팅 조합만 비교한다.

campaign_cases = [
    {
        "캠페인": "앱 푸시 · 할인 30%",
        "channel": "앱 푸시",
        "offer_name": "구독료 할인",
        "benefit_pct": 30,
        "duration": 7,
    },
    {
        "캠페인": "이메일 · 할인 30%",
        "channel": "이메일",
        "offer_name": "구독료 할인",
        "benefit_pct": 30,
        "duration": 7,
    },
    {
        "캠페인": "앱 푸시 · 무료 이용",
        "channel": "앱 푸시",
        "offer_name": "무료 이용 혜택",
        "benefit_pct": 0,
        "duration": 7,
    },
    {
        "캠페인": "앱 푸시 · 콘텐츠 추천",
        "channel": "앱 푸시",
        "offer_name": "콘텐츠 추천",
        "benefit_pct": 0,
        "duration": 7,
    },
]

In [166]:
campaign_results = {}
campaign_summaries = []

for segment_name in SEGMENTS.keys():
    for campaign in campaign_cases:
        result = simulate_segment(
            segment_name=segment_name,
            channel=campaign["channel"],
            offer_name=campaign["offer_name"],
            benefit_pct=campaign[
                "benefit_pct"
            ],
            duration=campaign["duration"],
        )

        result_key = (
            segment_name,
            campaign["캠페인"],
        )

        # 이후 특정 조합의 상세 결과를 확인할 수 있도록 저장
        campaign_results[
            result_key
        ] = result

        summary = result[
            "summary"
        ].copy()

        summary.insert(
            1,
            "캠페인",
            campaign["캠페인"],
        )

        campaign_summaries.append(
            summary
        )

campaign_comparison = pd.concat(
    campaign_summaries,
    ignore_index=True,
)

In [167]:
compact_columns = [
    "분류",
    "캠페인",
    "고객수",
    "평균_행동변화율(%)",
    "현재_평균이탈확률",
    "캠페인후_평균이탈확률",
    "예상_감소인원",
    "예상이탈_변화율(%)",
]

display(
    campaign_comparison[
        compact_columns
    ].round(
        {
            "평균_행동변화율(%)": 2,
            "현재_평균이탈확률": 4,
            "캠페인후_평균이탈확률": 4,
            "예상_감소인원": 2,
            "예상이탈_변화율(%)": 2,
        }
    )
)

,분류,캠페인,고객수,평균_행동변화율(%),현재_평균이탈확률,캠페인후_평균이탈확률,예상_감소인원,예상이탈_변화율(%)
0,자동갱신 끔 · 활동량 저하형,앱 푸시 · 할인 30%,2575,23.37,0.4779,0.4535,62.85,-5.11
1,자동갱신 끔 · 활동량 저하형,이메일 · 할인 30%,2575,10.21,0.4779,0.4674,27.05,-2.20
2,자동갱신 끔 · 활동량 저하형,앱 푸시 · 무료 이용,2575,24.84,0.4779,0.4516,67.71,-5.50
3,자동갱신 끔 · 활동량 저하형,앱 푸시 · 콘텐츠 추천,2575,19.15,0.4779,0.4570,53.87,-4.38
4,자동갱신 끔 · 결제·해지형,앱 푸시 · 할인 30%,2308,28.35,0.7427,0.7387,9.16,-0.53
5,자동갱신 끔 · 결제·해지형,이메일 · 할인 30%,2308,10.21,0.7427,0.7411,3.79,-0.22
6,자동갱신 끔 · 결제·해지형,앱 푸시 · 무료 이용,2308,30.49,0.7427,0.7387,9.13,-0.53
7,자동갱신 끔 · 결제·해지형,앱 푸시 · 콘텐츠 추천,2308,22.90,0.7427,0.7392,8.19,-0.48
8,자동갱신 끔 · 휴면형,앱 푸시 · 할인 30%,1307,24.44,0.5516,0.5329,24.46,-3.39
9,자동갱신 끔 · 휴면형,이메일 · 할인 30%,1307,10.05,0.5516,0.5431,11.12,-1.54


In [168]:
campaign_order = [
    campaign["캠페인"]
    for campaign in campaign_cases
]

churn_change_pivot = (
    campaign_comparison
    .pivot(
        index="분류",
        columns="캠페인",
        values="예상이탈_변화율(%)",
    )
    .reindex(
        index=list(SEGMENTS.keys()),
        columns=campaign_order,
    )
)

display(
    churn_change_pivot.round(2)
)

캠페인,앱 푸시 · 할인 30%,이메일 · 할인 30%,앱 푸시 · 무료 이용,앱 푸시 · 콘텐츠 추천
분류,,,,
자동갱신 끔 · 활동량 저하형,-5.11,-2.20,-5.50,-4.38
자동갱신 끔 · 결제·해지형,-0.53,-0.22,-0.53,-0.48
자동갱신 끔 · 휴면형,-3.39,-1.54,-3.51,-2.92
자동갱신 켬 · 결제·해지형,-1.21,-0.54,-1.31,-1.00
자동갱신 켬 · 휴면형,-0.36,-0.20,-0.40,-0.36


## 5. 시뮬레이션 결과 검증

### 5-1. 예상 감소 인원의 95% 범위

In [169]:
def boot_range(
    prob_before,
    prob_after,
    n_boot=1000,
    seed=42,
):
    """
    선택된 고객들을 반복해서 다시 뽑아
    예상 감소 인원의 95% 범위를 계산한다.

    n_boot : 반복 횟수
    seed   : 실행할 때마다 같은 결과가 나오도록 하는 값
    """

    rng = np.random.default_rng(
        seed
    )

    diff = (
        np.asarray(
            prob_before,
            dtype=float,
        )
        - np.asarray(
            prob_after,
            dtype=float,
        )
    )

    n = len(diff)

    if n == 0:
        return np.array(
            [np.nan, np.nan]
        )

    sums = np.array(
        [
            diff[
                rng.integers(
                    0,
                    n,
                    n,
                )
            ].sum()
            for _ in range(n_boot)
        ]
    )

    return np.percentile(
        sums,
        [2.5, 97.5],
    )

In [170]:
range_rows = []

for (
    segment_name,
    campaign_name,
), result in campaign_results.items():

    detail = result["detail"]

    range_low, range_high = boot_range(
        prob_before=detail[
            "prob_before"
        ],
        prob_after=detail[
            "prob_after"
        ],
        n_boot=1000,
        seed=42,
    )

    summary_row = (
        result["summary"].iloc[0]
    )

    range_rows.append(
        {
            "분류": segment_name,
            "캠페인": campaign_name,

            "예상_감소인원": (
                detail[
                    "churn_reduction"
                ].sum()
            ),

            "감소인원_하한(95%)": (
                range_low
            ),

            "감소인원_상한(95%)": (
                range_high
            ),

            "예상이탈_변화율(%)": (
                summary_row[
                    "예상이탈_변화율(%)"
                ]
            ),
        }
    )

simulation_range_table = pd.DataFrame(
    range_rows
)

display(
    simulation_range_table.round(
        {
            "예상_감소인원": 2,
            "감소인원_하한(95%)": 2,
            "감소인원_상한(95%)": 2,
            "예상이탈_변화율(%)": 2,
        }
    )
)

,분류,캠페인,예상_감소인원,감소인원_하한(95%),감소인원_상한(95%),예상이탈_변화율(%)
0,자동갱신 끔 · 활동량 저하형,앱 푸시 · 할인 30%,62.85,58.63,66.56,-5.11
1,자동갱신 끔 · 활동량 저하형,이메일 · 할인 30%,27.05,24.36,29.65,-2.20
2,자동갱신 끔 · 활동량 저하형,앱 푸시 · 무료 이용,67.71,63.55,71.81,-5.50
3,자동갱신 끔 · 활동량 저하형,앱 푸시 · 콘텐츠 추천,53.87,49.90,57.57,-4.38
4,자동갱신 끔 · 결제·해지형,앱 푸시 · 할인 30%,9.16,7.46,10.70,-0.53
5,자동갱신 끔 · 결제·해지형,이메일 · 할인 30%,3.79,2.70,4.91,-0.22
6,자동갱신 끔 · 결제·해지형,앱 푸시 · 무료 이용,9.13,7.44,10.67,-0.53
7,자동갱신 끔 · 결제·해지형,앱 푸시 · 콘텐츠 추천,8.19,6.64,9.73,-0.48
8,자동갱신 끔 · 휴면형,앱 푸시 · 할인 30%,24.46,21.85,27.22,-3.39
9,자동갱신 끔 · 휴면형,이메일 · 할인 30%,11.12,9.30,13.02,-1.54


In [171]:
churn_rate_before = simulation_detail["prob_before"].mean()
churn_rate_after = simulation_detail["prob_after"].mean()

churn_rate_point_change = (
    churn_rate_after - churn_rate_before
) * 100

expected_churn_before = simulation_detail["prob_before"].sum()
expected_churn_after = simulation_detail["prob_after"].sum()

reduced_customers = (
    expected_churn_before - expected_churn_after
)

churn_reduction_rate = (
    expected_churn_after / expected_churn_before - 1
) * 100

In [172]:
simulation_summary = pd.DataFrame(
    {
        "segment": [test_segment_name],
        "고객수": [len(simulation_detail)],
        "적용전_예상이탈률(%)": [churn_rate_before * 100],
        "적용후_예상이탈률(%)": [churn_rate_after * 100],
        "이탈률_변화(%p)": [churn_rate_point_change],
        "적용전_예상이탈자": [expected_churn_before],
        "적용후_예상이탈자": [expected_churn_after],
        "예상_감소인원": [reduced_customers],
        "상대_이탈감소율(%)": [churn_reduction_rate],
    }
)

display(simulation_summary.round(2))

,segment,고객수,적용전_예상이탈률(%),적용후_예상이탈률(%),이탈률_변화(%p),적용전_예상이탈자,적용후_예상이탈자,예상_감소인원,상대_이탈감소율(%)
0,자동갱신 끔 · 활동량 저하형,2575,47.79,45.35,-2.44,1230.61,1167.76,62.85,-5.11


In [7]:
import numpy as np
import pandas as pd

def sync_totals(X, orig):
    """하루 평균 × 활동일 = 총량이 되도록 다시 맞춘다 (활동일 0인 고객은 원래 값 유지)"""
    d = X["activity_days"]
    ok = d > 0
    X.loc[ok, "total_secs"] = X.loc[ok, "avg_daily_secs"] * d[ok]
    X.loc[ok, "total_num_unq"] = (X.loc[ok, "avg_daily_unq"] * d[ok]).round()
    X.loc[ok, "total_num_100"] = (X.loc[ok, "avg_daily_complete"] * d[ok]).round()
    X.loc[ok, "complete_per_unq"] = X.loc[ok, "total_num_100"] / X.loc[ok, "total_num_unq"].replace(0, np.nan)
    for c in ["total_secs", "total_num_unq", "total_num_100", "complete_per_unq"]:
        X.loc[~ok, c] = orig.loc[~ok, c]
    return X


def lever_listen_time(X, p):      # 하루 청취시간 ↑
    X["avg_daily_secs"] *= 1 + p
    return X

def lever_song_variety(X, p):     # 하루 고유곡 ↑
    X["avg_daily_unq"] *= 1 + p
    return X

def lever_completion(X, p):       # 완청 수준 ↑
    X["avg_daily_complete"] *= 1 + p
    return X

def lever_activity_days(X, p):    # 활동일 ↑
    X["activity_days"] = (X["activity_days"] * (1 + p)).clip(upper=90).round()
    return X

def lever_revisit(X, p):          # 마지막 접속 후 일수 ↓
    X["days_since_last_log"] = (X["days_since_last_log"] * (1 - p)).round()
    return X

def lever_plan_extend(X, p):      # 플랜 기간 ↑
    X["last_plan_days"] = (X["last_plan_days"] * (1 + p)).round()
    return X

def lever_cancel_stop(X, p):      # 해지 철회 (1 → 0)
    X.loc[X["last_is_cancel"] == 1, ["last_is_cancel", "cancel_on_last_date"]] = 0
    return X

def lever_auto_renew(X, p):       # 자동갱신 전환 (0 → 1)
    X.loc[X["last_auto_renew"] == 0, "last_auto_renew"] = 1
    return X


LEVERS = {
    "하루 청취시간 ↑": lever_listen_time,
    "하루 고유곡 ↑": lever_song_variety,
    "완청 수준 ↑": lever_completion,
    "활동일 ↑": lever_activity_days,
    "재접속 (접속 공백 ↓)": lever_revisit,
    "플랜 기간 ↑": lever_plan_extend,
    "해지 철회": lever_cancel_stop,
    "자동갱신 전환": lever_auto_renew,
}

In [8]:
X0 = risk_users[CHURN_FEATS].copy()
prob0 = churn_model.predict_proba(X0)[:, 1]

rows = []
for name, fn in LEVERS.items():
    X1 = fn(X0.copy(), 0.2)
    X1 = sync_totals(X1, X0)
    prob1 = churn_model.predict_proba(X1)[:, 1]
    diff = prob1 - prob0
    changed = np.abs(diff) > 1e-9
    rows.append({
        "레버": name,
        "평균 변화(%p)": diff.mean() * 100,
        "바뀐 고객 평균(%p)": diff[changed].mean() * 100 if changed.any() else 0,
        "바뀐 고객 수": int(changed.sum()),
        "확률 오른 비율(%)": (diff[changed] > 0).mean() * 100 if changed.any() else 0,
    })

check = pd.DataFrame(rows).round(3)
check

,레버,평균 변화(%p),바뀐 고객 평균(%p),바뀐 고객 수,확률 오른 비율(%)
0,하루 청취시간 ↑,-0.089,-0.255,2622,32.990
1,하루 고유곡 ↑,-0.627,-0.877,5368,36.233
2,완청 수준 ↑,0.367,0.587,4703,66.064
3,활동일 ↑,-1.088,-1.581,5172,29.196
4,재접속 (접속 공백 ↓),-0.194,-1.737,839,11.800
5,플랜 기간 ↑,3.705,5.412,5146,86.572
6,해지 철회,-9.824,-76.211,969,0.000
7,자동갱신 전환,-41.933,-50.922,6190,0.048


In [9]:
seg = risk_users["segment"].values
by_cluster = {}
for name, fn in LEVERS.items():
    X1 = sync_totals(fn(X0.copy(), 0.2), X0)
    diff = churn_model.predict_proba(X1)[:, 1] - prob0
    by_cluster[name] = pd.Series(diff * 100, index=risk_users.index).groupby(seg).mean()

pd.DataFrame(by_cluster).T.round(3)

,미분류,반복 거래·취소 위험형,장기 관계·고빈도 거래형,장기 플랜·고결제 고위험형,저활동·단기 구독형
하루 청취시간 ↑,-0.000,-0.063,-0.050,-0.114,-0.112
하루 고유곡 ↑,0.002,-0.989,-0.690,0.009,-1.012
완청 수준 ↑,0.010,0.701,0.336,0.150,0.504
활동일 ↑,0.003,-0.502,-0.815,-0.149,-2.002
재접속 (접속 공백 ↓),-0.000,-0.385,-0.142,-0.028,-0.299
플랜 기간 ↑,0.434,0.084,3.253,0.028,7.163
해지 철회,-35.032,-66.552,0.000,0.000,0.000
자동갱신 전환,-17.466,-0.360,-48.209,-76.537,-36.899
